# Evaluation: 5-Algorithm Comparison

Compares all five segmentation algorithms on the TACO test set (150 images, 3 classes).

| # | Algorithm | Type | Requires training |
|---|-----------|------|------------------|
| 1 | Distance-based Watershed | Pure CV | No |
| 2 | Morphology-guided Watershed | Pure CV | No |
| 3 | Faster R-CNN + Watershed | Detection + CV | Yes |
| 4 | YOLOv8-seg + Watershed | Detection + CV | Yes |
| 5 | RT-DETR + Watershed | Detection + CV | Yes |

**Run order:** `remap_classes.py` -> Algorithms 1-5 notebooks -> this notebook.

**Metrics:**
- `box_map50` / `mask_map50` -- COCO mAP@50 (N/A for pure-CV algorithms, shown as 0)
- `binary_iou` -- mean pixel-level IoU of predicted masks vs GT masks per image
- `counting_mae` -- mean |predicted count - GT count| per image
- `counting_within_1/3` -- % of images where count error <= 1 / <= 3
- `avg_inference_ms` -- mean wall-clock time per image

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
RUNS_DIR = os.path.join(BASE_DIR, "runs")

# ── Result file paths ────────────────────────────────────────────────────────
RESULT_FILES = {
    "Alg1 Distance-WS":   os.path.join(RUNS_DIR, "watershed_distance",  "watershed_distance_results.json"),
    "Alg2 Morphology-WS": os.path.join(RUNS_DIR, "watershed_morphology", "watershed_morphology_results.json"),
    "Alg3 FRCNN+WS":      os.path.join(RUNS_DIR, "faster_rcnn",          "fasterrcnn_watershed_results.json"),
    "Alg4 YOLO+WS":       os.path.join(RUNS_DIR, "yolov8_seg",           "yolo_watershed_results.json"),
    "Alg5 RTDETR+WS":     os.path.join(RUNS_DIR, "rtdetr",               "rtdetr_watershed_results.json"),
}

# ── Load results ─────────────────────────────────────────────────────────────
results = {}
for name, path in RESULT_FILES.items():
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
        print(f"OK      {name}")
    else:
        print(f"MISSING {name}  ->  {path}")

print(f"\nLoaded {len(results)}/5 result files.")

## Summary Table

In [ ]:
METRIC_KEYS = [
    "box_map50", "mask_map50", "binary_iou",
    "counting_mae", "counting_within_1", "counting_within_3",
    "avg_inference_ms",
]
METRIC_LABELS = [
    "Box mAP@50", "Mask mAP@50", "Binary IoU",
    "Count MAE", "Within+-1 (%)", "Within+-3 (%)",
    "Inf (ms)",
]

rows = []
for name, data in results.items():
    m = data["metrics"]
    row = {"Algorithm": name}
    for key, label in zip(METRIC_KEYS, METRIC_LABELS):
        val = m.get(key, 0.0)
        if key == "avg_inference_ms":
            row[label] = f"{val:.0f}"
        elif key in ("counting_within_1", "counting_within_3", "counting_mae"):
            row[label] = f"{val:.1f}"
        else:
            row[label] = f"{val:.3f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Algorithm")
print("\n" + "="*85)
print("ALGORITHM COMPARISON  (test set, 3-class TACO)")
print("="*85)
print(df.to_string())
print("="*85)
print("Note: Box/Mask mAP shown as 0.000 for pure-CV algorithms (Alg1, Alg2).")

## Bar Charts

In [ ]:
alg_names   = list(results.keys())
short_names = [n.split()[0] for n in alg_names]   # Alg1, Alg2, ...
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

def get_metric(name, key):
    return results[name]["metrics"].get(key, 0.0)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_specs = [
    ("mask_map50",        "Mask mAP@50",       True,  "higher = better; N/A for Alg1/2"),
    ("binary_iou",        "Binary IoU",         True,  "higher = better"),
    ("box_map50",         "Box mAP@50",         True,  "higher = better; N/A for Alg1/2"),
    ("counting_mae",      "Counting MAE",       False, "lower = better"),
    ("counting_within_1", "Within+-1 (%)",      True,  "higher = better"),
    ("avg_inference_ms",  "Avg Inference (ms)", False, "lower = better"),
]

for ax, (key, title, higher_better, note) in zip(axes.flat, plot_specs):
    vals = [get_metric(n, key) for n in alg_names]
    bars = ax.bar(short_names, vals, color=colors[:len(alg_names)])
    ax.set_title(f"{title}\n({note})", fontsize=9)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals) * 0.01 if max(vals) > 0 else 0.01,
                f"{v:.2f}", ha='center', va='bottom', fontsize=8)
    if higher_better and max(vals) <= 1.0:
        ax.set_ylim(0, 1.1)

plt.suptitle("5-Algorithm Comparison -- TACO 3-class Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "algorithm_comparison.png"), dpi=150)
plt.show()
print("Saved: runs/algorithm_comparison.png")

## Radar Chart

In [ ]:
radar_keys   = ["mask_map50", "binary_iou", "counting_within_1", "counting_within_3"]
radar_labels = ["Mask mAP@50", "Binary IoU", "Within+-1", "Within+-3"]

def norm_val(name, key):
    v = get_metric(name, key)
    if "within" in key:
        v /= 100.0
    return v

N      = len(radar_keys)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for name, color in zip(alg_names, colors):
    vals = [norm_val(name, k) for k in radar_keys]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.0"], size=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.set_title("Algorithm Radar (normalised, higher = better)",
             pad=20, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "algorithm_radar.png"), dpi=150)
plt.show()
print("Saved: runs/algorithm_radar.png")

## Counting Error Distribution

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4), sharey=True)

for ax, (name, data), color in zip(axes, results.items(), colors):
    errors = [r["count_error"] for r in data.get("per_image", [])]
    if not errors:
        ax.set_title(name.split()[0] + "\nNo data"); ax.axis("off"); continue
    bins = range(0, max(errors) + 2)
    ax.hist(errors, bins=bins, color=color, alpha=0.8, edgecolor='white')
    mae = np.mean(errors)
    ax.axvline(mae, color='red', linestyle='--', linewidth=1.5, label=f"MAE={mae:.1f}")
    ax.set_title(name.split()[0], fontsize=10)
    ax.set_xlabel("Count error")
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel("# images")
plt.suptitle("Counting Error Distribution per Algorithm", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RUNS_DIR, "counting_error_distribution.png"), dpi=150)
plt.show()
print("Saved: runs/counting_error_distribution.png")

## Weighted Scoring & Recommendation

In [ ]:
WEIGHTS = {
    "mask_map50":        0.30,
    "binary_iou":        0.25,
    "counting_within_1": 0.20,
    "counting_mae":      0.15,  # lower=better: inverted
    "avg_inference_ms":  0.10,  # lower=better: inverted
}

raw_vals = {key: np.array([get_metric(n, key) for n in alg_names]) for key in WEIGHTS}

def normalise(arr, invert=False):
    rng = arr.max() - arr.min()
    if rng == 0:
        return np.ones_like(arr) * 0.5
    n = (arr - arr.min()) / rng
    return 1 - n if invert else n

scored = np.zeros(len(alg_names))
for key, w in WEIGHTS.items():
    invert = key in ("counting_mae", "avg_inference_ms")
    scored += w * normalise(raw_vals[key], invert=invert)

score_df = pd.DataFrame({
    "Algorithm":      alg_names,
    "Weighted Score": scored.round(4),
}).sort_values("Weighted Score", ascending=False).reset_index(drop=True)

print("\n" + "="*50)
print("WEIGHTED RANKING")
print("(mask_map=0.30, binary_iou=0.25, within+-1=0.20,")
print(" count_mae=0.15, speed=0.10)")
print("="*50)
print(score_df.to_string(index=False))
print("="*50)

winner = score_df.iloc[0]["Algorithm"]
print(f"\nRecommended algorithm: {winner}")

# Save final report
report = {
    "recommendation":  winner,
    "scores":          dict(zip(alg_names, scored.tolist())),
    "metrics_summary": {name: data["metrics"] for name, data in results.items()},
}
out_path = os.path.join(RUNS_DIR, "evaluation_report.json")
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"Saved: {out_path}")